# RAG Prompt Compression

- RAG에서 검색된 문서 컨텍스트를 LLM에 넣기 전에 압축(요약/추출)해서 프롬프트를 가볍게 만드는 단계(또는 그 함수/체인)”
    - rag: Retrieval-Augmented Generation(검색 기반 생성)
    - prompt: LLM에 넣는 입력(프롬프트/컨텍스트)
    - compression: 길이를 줄이거나 핵심만 남기는 처리(요약, 중요 문장 추출, 중복 제거 등)

- rag_prompt_compression의 목적
    - 토큰 제한(컨텍스트 윈도우) 안에 더 많은 핵심정보를 넣기
    - 비용/지연 줄이기(입력 토큰 감소)
    - 불필요한 문장 제거로 답변 품질 안정화

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

In [ ]:
from langchain_core.documents import Document

# 벡터DB와 같은 형식의 결과를 반환하는 더미 리트리버 함수
def retrieve_vectordb(query=None):
    return [
        Document(metadata = {'source': 'doc1'}, page_content='현대 교육 분야에서는 인공지능(AI) 기반 플랫폼이 단순한 보조 도구를 넘어 학습 전반을 설계하는 핵심 인프라로 자리 잡고 있다. 초기 AI 교육 시스템은 출결 관리, 문제 자동 채점, 진도 체크처럼 교사의 업무 부담을 줄이는 기능에 집중했으나, 최근에는 학습자의 행동 로그, 성취도, 반복 오류 패턴 등을 분석해 개인별 학습 성향을 파악하는 방향으로 발전했다. 이를 통해 학습자마다 난이도, 콘텐츠 유형, 학습 속도를 다르게 조정하는 맞춤형 학습 추천 시스템이 구현되고 있다. 더 나아가 자연어 처리(NLP) 기술의 발전으로 강의 영상이나 교재를 자동 요약하거나 핵심 개념을 재구성해 제공하는 기능이 가능해졌으며, 학습자가 질문을 입력하면 실시간으로 맥락을 이해하고 답변하는 챗봇 튜터가 등장해 자기주도 학습을 강화하고 있다. 이러한 변화는 교육을 일방향 전달에서 상호작용 기반 학습으로 전환시키는 중요한 역할을 하고 있다.'),
        Document(metadata = {'source': 'doc2'}, page_content='Retrieval-Augmented Generation(RAG)은 대규모 언어 모델(LLM)이 가진 언어 생성 능력에 외부 지식 저장소를 결합함으로써, 모델이 학습하지 않았거나 최신성이 필요한 정보까지 활용할 수 있도록 한 구조이다. 기존 LLM은 학습 시점 이후의 정보를 반영하기 어렵고, 사실과 다른 내용을 생성하는 문제가 있었으나, RAG는 벡터 DB에 저장된 문서를 검색해 이를 보완한다. RAG 파이프라인은 일반적으로 세 단계로 구성되며, 먼저 사용자의 질의를 임베딩 벡터로 변환해 의미적으로 가장 유사한 문서를 Top-K 방식으로 검색한다. 이후 검색된 문서들을 프롬프트 컨텍스트로 구성해 LLM에 전달하고, 마지막으로 LLM이 해당 정보를 종합·추론해 최종 답변을 생성한다. 이 구조를 통해 답변의 정확성과 근거성이 향상되며, 기업 문서 검색, 사내 지식 Q&A, 교육용 튜터 시스템 등 다양한 실무 환경에서 활용되고 있다.'),
        Document(metadata = {'source': 'doc3'}, page_content='프롬프트 압축(Prompt Compression)은 LLM에 전달되는 입력 컨텍스트를 최대한 간결하게 정제하여, 제한된 토큰 수 안에서 핵심 정보만 효과적으로 전달하기 위한 기법이다. LLM은 입력 토큰 수에 따라 비용과 응답 시간이 증가하기 때문에, 불필요한 문장이나 중복된 설명이 많을수록 시스템 효율이 급격히 떨어진다. 이를 해결하기 위해 프롬프트 압축은 문서 요약, 중요 문장 추출, 의미 중복 제거, 구조화된 키포인트 변환 등의 방법을 사용한다. 특히 RAG 환경에서는 검색된 여러 문서를 그대로 넣는 대신 핵심 정보만 압축해 전달함으로써, 답변 품질을 유지하면서도 비용과 지연을 줄일 수 있다. 결과적으로 프롬프트 압축은 대규모 LLM 서비스를 안정적으로 운영하기 위한 필수 최적화 전략으로 활용되고 있다.'),
    ]

retrieve_vectordb()

[Document(metadata={'source': 'doc1'}, page_content='현대 교육 분야에서는 인공지능(AI) 기반 플랫폼이 단순한 보조 도구를 넘어 학습 전반을 설계하는 핵심 인프라로 자리 잡고 있다. 초기 AI 교육 시스템은 출결 관리, 문제 자동 채점, 진도 체크처럼 교사의 업무 부담을 줄이는 기능에 집중했으나, 최근에는 학습자의 행동 로그, 성취도, 반복 오류 패턴 등을 분석해 개인별 학습 성향을 파악하는 방향으로 발전했다. 이를 통해 학습자마다 난이도, 콘텐츠 유형, 학습 속도를 다르게 조정하는 맞춤형 학습 추천 시스템이 구현되고 있다. 더 나아가 자연어 처리(NLP) 기술의 발전으로 강의 영상이나 교재를 자동 요약하거나 핵심 개념을 재구성해 제공하는 기능이 가능해졌으며, 학습자가 질문을 입력하면 실시간으로 맥락을 이해하고 답변하는 챗봇 튜터가 등장해 자기주도 학습을 강화하고 있다. 이러한 변화는 교육을 일방향 전달에서 상호작용 기반 학습으로 전환시키는 중요한 역할을 하고 있다.'),
 Document(metadata={'source': 'doc2'}, page_content='Retrieval-Augmented Generation(RAG)은 대규모 언어 모델(LLM)이 가진 언어 생성 능력에 외부 지식 저장소를 결합함으로써, 모델이 학습하지 않았거나 최신성이 필요한 정보까지 활용할 수 있도록 한 구조이다. 기존 LLM은 학습 시점 이후의 정보를 반영하기 어렵고, 사실과 다른 내용을 생성하는 문제가 있었으나, RAG는 벡터 DB에 저장된 문서를 검색해 이를 보완한다. RAG 파이프라인은 일반적으로 세 단계로 구성되며, 먼저 사용자의 질의를 임베딩 벡터로 변환해 의미적으로 가장 유사한 문서를 Top-K 방식으로 검색한다. 이후 검색된 문서들을 프롬프트 컨텍스트로 구성해 LLM에 전달하고, 마지막으로 LLM이 해당 정보를 종합·추론해 최종 답변을 생성한다. 이 구조를 통해 답변의 정확성과 근거성이 향상되며, 기업 문서 검색, 사내 지

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model('gpt-5.6-luna', n = 5) # 한 번 요청으로 5개 응답 생성
prompt = ChatPromptTemplate.from_template('''
다음 문서의 내용을 300토큰 이내로 요약해 주세요.

{docs}
''')

output_parser = StrOutputParser()

summary_chain = prompt | llm | output_parser


retrieved_docs = retrieve_vectordb() # 더미데이터 생성 [Document, ...]
context = '\n\n'.join([doc.page_content for doc in retrieved_docs])

compressed_context = summary_chain.invoke({'docs': context})
print(compressed_context)

AI 교육 플랫폼은 출결·채점 중심에서 학습자의 행동 로그와 성취도, 오류 패턴을 분석해 난이도·콘텐츠·속도를 개인화하는 맞춤형 시스템으로 발전했다. NLP를 활용한 자동 요약과 챗봇 튜터는 자기주도 학습과 상호작용을 강화한다.

RAG는 LLM에 외부 지식 저장소를 결합해 최신성과 정확성을 높이는 구조다. 질의를 임베딩해 관련 문서를 검색하고, 이를 컨텍스트로 제공한 뒤 LLM이 답변을 생성한다. 교육용 튜터와 문서 검색 등 다양한 분야에 활용된다.

프롬프트 압축은 요약·중요 문장 추출·중복 제거로 입력 컨텍스트를 간결화하는 기법이다. 특히 RAG에서 핵심 정보만 전달해 토큰 비용과 응답 지연을 줄이면서 답변 품질과 운영 효율을 높인다.


In [4]:
question = 'RAG가 그래서 뭐죠?'
llm = init_chat_model('gpt-5.6-luna', n = 5) # 한 번 요청으로 5개 응답 생성
prompt = ChatPromptTemplate.from_template('''
[조회된 문서]를 기반으로 사용자의 [질문]에 성실하게 답변해주세요.

[조회된 문서]
{context}

[질문]
{question}
''')

output_parser = StrOutputParser()

rag_chain = prompt | llm | output_parser

print(rag_chain.invoke({'context': compressed_context,'question':question}))


RAG는 **검색 증강 생성(Retrieval-Augmented Generation)**의 약자로, AI가 답변하기 전에 외부 문서나 지식 저장소에서 관련 정보를 검색해 참고하도록 하는 방식입니다.

작동 과정은 간단합니다.

1. 사용자의 질문을 분석하고 검색하기 좋은 형태로 바꿉니다.
2. 관련 문서를 지식 저장소에서 찾습니다.
3. 검색한 내용을 LLM에 참고 자료로 제공합니다.
4. LLM이 그 자료를 바탕으로 답변을 생성합니다.

예를 들어 교육용 AI 튜터라면, 학생의 질문과 관련된 교재·강의자료·학습 기록을 먼저 검색한 뒤 그 내용을 바탕으로 설명할 수 있습니다.

일반적인 LLM만 사용할 때보다 **최신 정보와 특정 기관의 문서**를 반영하기 쉽고, 근거 있는 답변을 생성하는 데 유리합니다. 또한 프롬프트 압축을 함께 사용하면 검색된 자료 중 핵심 내용만 전달해 **토큰 비용과 응답 시간을 줄일 수 있습니다.**


In [5]:
# 통합
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

rag_llm = init_chat_model('gpt-5.6-luna', n = 5) # 한 번 요청으로 5개 응답 생성
prompt = ChatPromptTemplate.from_template('''
[조회된 문서]를 기반으로 사용자의 [질문]에 성실하게 답변해주세요.

[조회된 문서]
{context}

[질문]
{question}
''')

rag_chain = prompt | rag_llm | output_parser

def rag(question: str):
    retrieved_docs = retrieve_vectordb() # 더미데이터 생성 [Document, ...]
    context = '\n\n'.join([doc.page_content for doc in retrieved_docs])

    compressed_context = summary_chain.invoke({'docs': context})
    return rag_chain.invoke({'context': compressed_context,'question':question})

question = 'RAG가 뭐에요?'
response = rag(question)
print(response)

RAG는 **검색 증강 생성(Retrieval-Augmented Generation)**의 약자입니다.

LLM이 답변을 만들기 전에 외부 지식 저장소나 문서 데이터베이스에서 질문과 관련된 정보를 검색하고, 검색한 내용을 참고해 답변을 생성하는 방식입니다.

일반적인 과정은 다음과 같습니다.

1. 사용자의 질문을 임베딩으로 변환
2. 벡터 데이터베이스에서 관련 문서 검색
3. 검색된 문서 중 핵심 내용을 LLM에 전달
4. LLM이 해당 정보를 근거로 답변 생성

이를 통해 모델이 학습하지 않은 최신 정보도 활용할 수 있고, 답변의 **정확성·근거성**을 높일 수 있습니다. 예를 들어 교육용 AI 튜터가 교과서나 강의 자료를 검색해 학생의 질문에 답하는 데 활용할 수 있습니다.


- **사내 문서 검색형 챗봇**에서 가장 많이 사용되는 RAG 기본 패턴 실습 (규정, 매뉴얼, 기술 문서, 교육 자료 등)
- Prompt Compression을 적용하면 **문서량이 많아도 토큰 한도를 안정적으로 관리**할 수 있어, 대규모 서비스 운영 시 **비용 절감과 응답 속도 개선**에 직접적인 효과가 있다.
- 요약 → RAG 응답을 분리한 구조는  
  - 요약 캐싱  
  - 문서 업데이트 시 재요약  
  - 사용자 질문 유형별 요약 전략 분리  
  등 **운영 관점의 확장성**이 높다.
- 실제 실무에서는 이 파이프라인을  
  - 고객센터 FAQ 자동 응답  
  - 내부 지식 Q&A 시스템  
  - 교육/온보딩 튜터  
  - 법무·의료 문서 보조 분석  
  등에 그대로 적용할 수 있다.